In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Load environment variables from .env file
load_dotenv()

# Get URLs from environment
postgres_url = os.getenv("POSTGRES_URL")
mysql_url = os.getenv("MYSQL_URL")

# Create engines securely
pg_engine = create_engine(postgres_url)
mysql_engine = create_engine(mysql_url)

In [3]:
print(pd.read_sql('SELECT 1 AS test', pg_engine))
print(pd.read_sql('SELECT 1 AS test', mysql_engine))

   test
0     1
   test
0     1


In [4]:
pg_tables = pd.read_sql("SELECT tablename FROM pg_tables WHERE schemaname='public'", pg_engine)
mysql_tables = pd.read_sql("SHOW TABLES", mysql_engine)
print(pg_tables)
print(mysql_tables)

        tablename
0  routes_details
1  routes_weather
      Tables_in_maindb
0         city_weather
1      drivers_details
2      traffic_details
3        truck_details
4  truck_schedule_data


In [5]:
df_sample = pd.read_sql('SELECT * FROM routes_weather LIMIT 1000;', pg_engine)
df_sample.head()

,route_id,Date,temp,wind_speed,description,precip,humidity,visibility,pressure,chanceofrain,chanceoffog,chanceofsnow,chanceofthunder
0,R-ada2a391,2019-01-01 00:00:00,30.0,11.0,Heavy snow,0.0,90,1.0,1010,0,0,0,0
1,R-ada2a391,2019-01-01 06:00:00,30.0,11.0,Heavy snow,0.0,91,3.0,1012,0,0,0,0
2,R-ada2a391,2019-01-02 00:00:00,28.0,11.0,Cloudy,0.0,91,4.0,1013,0,0,0,0
3,R-ada2a391,2019-01-02 06:00:00,27.0,11.0,Cloudy,0.0,92,6.0,1015,0,0,0,0
4,R-ada2a391,2019-01-03 00:00:00,27.0,9.0,Cloudy,0.0,93,6.0,1016,0,0,0,0


In [6]:
df_sample=pd.read_sql('SELECT * FROM city_weather LIMIT 1000;',mysql_engine)
df_sample.head()

,city_id,date,hour,temp,wind_speed,description,precip,humidity,visibility,pressure,chanceofrain,chanceoffog,chanceofsnow,chanceofthunder
0,C-927ceb5e,2019-01-01,0,30.0,11.0,Light snow,0.0,86,6.0,1019.0,0.0,0.0,0.0,0.0
1,C-927ceb5e,2019-01-01,100,28.0,12.0,Light snow,0.0,86,5.0,1021.0,0.0,0.0,0.0,0.0
2,C-927ceb5e,2019-01-01,200,28.0,13.0,Moderate snow,0.0,85,4.0,1022.0,0.0,0.0,0.0,0.0
3,C-927ceb5e,2019-01-01,300,28.0,14.0,Moderate snow,0.0,84,3.0,1024.0,0.0,0.0,0.0,0.0
4,C-927ceb5e,2019-01-01,400,28.0,13.0,Moderate snow,0.0,84,3.0,1025.0,0.0,0.0,0.0,0.0


In [7]:
routes_df = pd.read_sql('SELECT * FROM routes_details', pg_engine)
routes_weather_df = pd.read_sql('SELECT * FROM routes_weather', pg_engine)

city_weather_df = pd.read_sql('SELECT * FROM city_weather', mysql_engine)
drivers_df = pd.read_sql('SELECT * FROM drivers_details', mysql_engine)
traffic_df = pd.read_sql('SELECT * FROM traffic_details', mysql_engine)
trucks_df = pd.read_sql('SELECT * FROM truck_details', mysql_engine)
truck_schedule_df = pd.read_sql('SELECT * FROM truck_schedule_data', mysql_engine)

In [8]:
dfs = {
    'routes': routes_df, 'routes_weather': routes_weather_df,
    'city_weather': city_weather_df, 'drivers': drivers_df,
    'traffic': traffic_df, 'trucks': trucks_df,
    'truck_schedule': truck_schedule_df
}
for name, df in dfs.items():
    print(name, df.shape)
    print(df.head(2))
    print('---')

routes (2352, 5)
     route_id   origin_id destination_id  distance  average_hours
0  R-ada2a391  C-927ceb5e     C-56e39a5e   1735.06          34.70
1  R-ae0ef31f  C-927ceb5e     C-73ae5412   1498.24          29.96
---
routes_weather (425712, 13)
     route_id                Date  temp  wind_speed description  precip  \
0  R-ada2a391 2019-01-01 00:00:00  30.0        11.0  Heavy snow     0.0   
1  R-ada2a391 2019-01-01 06:00:00  30.0        11.0  Heavy snow     0.0   

   humidity  visibility  pressure  chanceofrain  chanceoffog  chanceofsnow  \
0        90         1.0      1010             0            0             0   
1        91         3.0      1012             0            0             0   

   chanceofthunder  
0                0  
1                0  
---
city_weather (55176, 14)
      city_id        date  hour  temp  wind_speed description  precip  \
0  C-927ceb5e  2019-01-01     0  30.0        11.0  Light snow     0.0   
1  C-927ceb5e  2019-01-01   100  28.0        12.0  Lig

In [9]:
for name, df in dfs.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"\n{name}:")
    print(missing if not missing.empty else "No missing values")


routes:
No missing values

routes_weather:
No missing values

city_weather:
No missing values

drivers:
gender           23
driving_style    52
dtype: int64

traffic:
no_of_vehicles    1152
dtype: int64

trucks:
load_capacity_pounds    57
dtype: int64

truck_schedule:
No missing values


In [12]:
for name, df in dfs.items():
    print(f"\n{'='*50}\n{name} — shape: {df.shape}\n{'='*50}")
    print(df.dtypes)
    display(df.head())


routes — shape: (2352, 5)
route_id           object
origin_id          object
destination_id     object
distance          float64
average_hours     float64
dtype: object


,route_id,origin_id,destination_id,distance,average_hours
0,R-ada2a391,C-927ceb5e,C-56e39a5e,1735.06,34.70
1,R-ae0ef31f,C-927ceb5e,C-73ae5412,1498.24,29.96
2,R-4beec5fd,C-927ceb5e,C-4fe0fa24,6078.46,121.57
3,R-8d7a7fb2,C-927ceb5e,C-451776b7,1543.01,30.86
4,R-b236e347,C-927ceb5e,C-d80a1e7d,310.75,6.22



routes_weather — shape: (425712, 13)
route_id                   object
Date               datetime64[ns]
temp                      float64
wind_speed                float64
description                object
precip                    float64
humidity                    int64
visibility                float64
pressure                    int64
chanceofrain                int64
chanceoffog                 int64
chanceofsnow                int64
chanceofthunder             int64
dtype: object


,route_id,Date,temp,wind_speed,description,precip,humidity,visibility,pressure,chanceofrain,chanceoffog,chanceofsnow,chanceofthunder
0,R-ada2a391,2019-01-01 00:00:00,30.0,11.0,Heavy snow,0.0,90,1.0,1010,0,0,0,0
1,R-ada2a391,2019-01-01 06:00:00,30.0,11.0,Heavy snow,0.0,91,3.0,1012,0,0,0,0
2,R-ada2a391,2019-01-02 00:00:00,28.0,11.0,Cloudy,0.0,91,4.0,1013,0,0,0,0
3,R-ada2a391,2019-01-02 06:00:00,27.0,11.0,Cloudy,0.0,92,6.0,1015,0,0,0,0
4,R-ada2a391,2019-01-03 00:00:00,27.0,9.0,Cloudy,0.0,93,6.0,1016,0,0,0,0



city_weather — shape: (55176, 14)
city_id             object
date                object
hour                 int64
temp               float64
wind_speed         float64
description         object
precip             float64
humidity             int64
visibility         float64
pressure           float64
chanceofrain       float64
chanceoffog        float64
chanceofsnow       float64
chanceofthunder    float64
dtype: object


...

,route_id,date,hour,no_of_vehicles,accident
0,R-ada2a391,2019-01-01,0,669.0,0
1,R-ada2a391,2019-01-01,100,628.0,0
2,R-ada2a391,2019-01-01,200,516.0,0
3,R-ada2a391,2019-01-01,300,582.0,0
4,R-ada2a391,2019-01-01,400,564.0,0



trucks — shape: (1300, 5)
truck_id                  int64
truck_age                 int64
load_capacity_pounds    float64
mileage_mpg               int64
fuel_type                object
dtype: object


,truck_id,truck_age,load_capacity_pounds,mileage_mpg,fuel_type
0,42302347,10,3000.0,17,gas
1,27867488,14,10000.0,22,diesel
2,13927774,8,10000.0,19,gas
3,69577118,8,20000.0,19,gas
4,28650047,10,4000.0,21,diesel



truck_schedule — shape: (12308, 5)
truck_id                      int64
route_id                     object
departure_date       datetime64[ns]
estimated_arrival            object
delay                         int64
dtype: object


,truck_id,route_id,departure_date,estimated_arrival,delay
0,30312694,R-b236e347,2019-01-01 07:00:00,2019-01-01 13:13:12.,0
1,59856374,R-29ea762e,2019-01-01 07:00:00,2019-01-02 04:01:12.,0
2,12602955,R-a3d67783,2019-01-01 07:00:00,2019-01-01 07:45:36.,0
3,46619422,R-31ec9310,2019-01-01 07:00:00,2019-01-01 20:46:48.,0
4,10140178,R-a07c5dbd,2019-01-01 07:00:00,2019-01-01 21:34:11.,0


In [11]:
# List all dictionary variables currently in memory
%who dict

dfs	 
